In [4]:
!pip install spacy langdetect --quiet
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import pandas as pd
import spacy
import re
import logging
from collections import Counter

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

nlp = spacy.load('en_core_web_sm')
nlp.max_length = 2_000_000

print(" All imports done. spaCy model loaded.")

 All imports done. spaCy model loaded.


# Configuration

In [2]:
CONFIG = {
    'input_file'            : 'climate_abstracts.csv',
    'year_min'              : 2010,
    'year_max'              : 2025,
    'min_climate_terms'     : 1,
    'min_sentence_length'   : 40,
    'max_sentence_length'   : 600,
    'min_token_count'       : 8,
    'annotation_sample_size': 500,
    'sentences_file'        : 'climate_sentences.csv',
    'annotation_file'       : 'annotation_sample_500.csv',
}

CLIMATE_TERMS = [
    'CO2', 'carbon', 'climate', 'temperature', 'emission', 'greenhouse',
    'warming', 'sea level', 'glacier', 'ecosystem', 'biodiversity',
    'methane', 'arctic', 'deforestation', 'coral', 'drought', 'wildfire',
    'ocean', 'atmosphere', 'species', 'habitat', 'permafrost', 'flooding'
]

print("Configuration set.")

Configuration set.


# Loading CSV

In [3]:
df = pd.read_csv(CONFIG['input_file'])

print(f"Loaded       : {len(df)} abstracts")
print(f"Columns      : {list(df.columns)}")
print(f"Year range   : {df['year'].min()} – {df['year'].max()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
df.head(3)

Loaded       : 601 abstracts
Columns      : ['title', 'abstract', 'year', 'authors', 'venue', 'citation_count', 'fields']
Year range   : 2006.0 – 2026.0
Missing abstracts: 0


,title,abstract,year,authors,venue,citation_count,fields
0,Episodic vs. Sea Level Rise Coastal Flooding S...,Sea level rise (SLR) and increased urbanisatio...,2025.0,"Sebastian Spadotto, Saverio Fracaros, A. Bezzi...",Water,0,"Environmental Science, Geography"
1,Future Coastal Population Growth and Exposure ...,Coastal zones are exposed to a range of coasta...,2015.0,"B. Neumann, A. Vafeidis, J. Zimmermann, R. Nic...",PLoS ONE,2301,"Medicine, Geography, Environmental Science, Ge..."
2,New elevation data triple estimates of global ...,Most estimates of global mean sea-level rise t...,2019.0,"S. Kulp, B. Strauss",Nature Communications,1037,"Environmental Science, Medicine, Environmental..."


# Filtering low-quality papers

In [5]:
original_count = len(df)

# Fix year filter bug
df = df[(df['year'] >= CONFIG['year_min']) & (df['year'] <= CONFIG['year_max'])]
print(f"After year filter     : {len(df)} abstracts")

# Drop missing abstracts
df = df[df['abstract'].notna() & (df['abstract'].str.strip() != '')]
print(f"After dropping empty  : {len(df)} abstracts")

# Climate relevance filter
def count_climate_terms(text):
    text_lower = str(text).lower()
    return sum(1 for term in CLIMATE_TERMS if term.lower() in text_lower)

df['climate_term_count'] = df['abstract'].apply(count_climate_terms)
df = df[df['climate_term_count'] >= CONFIG['min_climate_terms']]
print(f"After relevance filter: {len(df)} abstracts")

df = df.reset_index(drop=True)
print(f"\nRemoved {original_count - len(df)} papers. {len(df)} clean abstracts remaining.")

After year filter     : 592 abstracts
After dropping empty  : 592 abstracts
After relevance filter: 592 abstracts

Removed 0 papers. 592 clean abstracts remaining.


# Dataset statistics

In [6]:
print(f"Total abstracts : {len(df)}")
print(f"Year range      : {int(df['year'].min())} – {int(df['year'].max())}")
print(f"Avg citations   : {df['citation_count'].mean():.1f}")
print(f"Unique venues   : {df['venue'].nunique()}")

print("\n--- Year Distribution ---")
print(df['year'].value_counts().sort_index().to_string())

print("\n--- Top 5 Journals ---")
print(df['venue'].value_counts().head(5).to_string())

Total abstracts : 592
Year range      : 2010 – 2025
Avg citations   : 95.9
Unique venues   : 219

--- Year Distribution ---
year
2010.0     1
2011.0     2
2012.0     6
2013.0    10
2014.0     9
2015.0     9
2016.0    17
2017.0    39
2018.0    39
2019.0    65
2020.0    78
2021.0    78
2022.0    61
2023.0    82
2024.0    64
2025.0    32

--- Top 5 Journals ---
venue
Scientific Reports                                                                 29
Global Change Biology                                                              29
Nature Communications                                                              27
Environmental Research Letters                                                     22
Proceedings of the National Academy of Sciences of the United States of America    14


# Sentence tokenization

In [7]:
def split_into_sentences(text):
    doc = nlp(text)
    sentences = []
    for sent in doc.sents:
        s = sent.text.strip()
        if len(s) < CONFIG['min_sentence_length']:   continue
        if len(s) > CONFIG['max_sentence_length']:   continue
        if len(s.split()) < CONFIG['min_token_count']: continue
        alpha_ratio = sum(c.isalpha() for c in s) / len(s)
        if alpha_ratio < 0.6: continue
        sentences.append(s)
    return sentences

print("Splitting abstracts into sentences")

all_sentences = []
for idx, row in df.iterrows():
    sentences = split_into_sentences(row['abstract'])
    for sent in sentences:
        all_sentences.append({
            'sentence_id'   : f"s{idx:04d}_{len(all_sentences):05d}",
            'sentence'      : sent,
            'source_title'  : row['title'],
            'year'          : row['year'],
            'venue'         : row['venue'],
            'citation_count': row['citation_count'],
            'fields'        : row['fields'],
        })

df_sentences = pd.DataFrame(all_sentences)

print(f"\nDone!")
print(f"Total sentences extracted  : {len(df_sentences)}")
print(f"Avg sentences per abstract : {len(df_sentences)/len(df):.1f}")
df_sentences.head(5)

Splitting abstracts into sentences

Done!
Total sentences extracted  : 5367
Avg sentences per abstract : 9.1


,sentence_id,sentence,source_title,year,venue,citation_count,fields
0,s0000_00000,Sea level rise (SLR) and increased urbanisatio...,Episodic vs. Sea Level Rise Coastal Flooding S...,2025.0,Water,0,"Environmental Science, Geography"
1,s0000_00001,"In this context, the role of hard coastal defe...",Episodic vs. Sea Level Rise Coastal Flooding S...,2025.0,Water,0,"Environmental Science, Geography"
2,s0000_00002,"Here, a thorough investigation is conducted in...",Episodic vs. Sea Level Rise Coastal Flooding S...,2025.0,Water,0,"Environmental Science, Geography"
3,s0000_00003,Grado is located on a barrier island of the ho...,Episodic vs. Sea Level Rise Coastal Flooding S...,2025.0,Water,0,"Environmental Science, Geography"
4,s0000_00004,The mean and maximum sea levels from the histo...,Episodic vs. Sea Level Rise Coastal Flooding S...,2025.0,Water,0,"Environmental Science, Geography"


# 500-sentence annotation sample

In [9]:
def score_sentence(text):
    text_lower = text.lower()
    return sum(1 for term in CLIMATE_TERMS if term.lower() in text_lower)

df_sentences['climate_score'] = df_sentences['sentence'].apply(score_sentence)

# Keep only sentences with at least 1 climate term
df_relevant = df_sentences[df_sentences['climate_score'] >= 1].copy()
print(f"Sentences with climate terms: {len(df_relevant)}")

# Stratified sample across time periods
df_relevant['year_bucket'] = df_relevant['year'].apply(
    lambda y: '2010-2014' if y <= 2014 else ('2015-2019' if y <= 2019 else '2020-2025')
)

sample_size = CONFIG['annotation_sample_size']
per_bucket  = sample_size // 3  # ~166 per time period

# Sample from each bucket separately, then combine
annotation_sample = (
    df_relevant
    .groupby('year_bucket', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), per_bucket), random_state=42))
)

# If we got fewer than 500, top up from remaining sentences
if len(annotation_sample) < sample_size:
    already_selected = annotation_sample.index
    remaining = df_relevant.drop(index=already_selected)
    needed = sample_size - len(annotation_sample)
    if len(remaining) >= needed:
        top_up = remaining.sample(needed, random_state=42)
        annotation_sample = pd.concat([annotation_sample, top_up])

annotation_sample = annotation_sample.sort_values('climate_score', ascending=False)
annotation_sample = annotation_sample.reset_index(drop=True)

# Add empty annotation columns
annotation_sample['entities']  = ''
annotation_sample['relations'] = ''
annotation_sample['annotated'] = False

print(f"\nAnnotation sample ready: {len(annotation_sample)} sentences")
print("\nYear bucket distribution:")
print(annotation_sample['year_bucket'].value_counts().to_string())
annotation_sample.head(5)

Sentences with climate terms: 3774

Annotation sample ready: 500 sentences

Year bucket distribution:
year_bucket
2020-2025    168
2010-2014    166
2015-2019    166


/tmp/ipykernel_3406/505014691.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), per_bucket), random_state=42))


,sentence_id,sentence,source_title,year,venue,citation_count,fields,climate_score,year_bucket,entities,relations,annotated
0,s0311_02805,This estimated net C transfer to the atmospher...,The impacts of recent permafrost thaw on land–...,2014.0,NaN,97,"Environmental Science, Physics, Environmental ...",6,2010-2014,,,False
1,s0453_04193,This information on the spatial and temporal p...,Groundwater discharge as a driver of methane e...,2022.0,Nature Communications,50,"Medicine, Environmental Science",5,2020-2025,,,False
2,s0461_04255,Net carbon dioxide (CO2) and methane (CH4) flu...,Permafrost thaw driven changes in hydrology an...,2021.0,Philosophical Transactions of the Royal Society A,49,"Medicine, Environmental Science, Geology",5,2020-2025,,,False
3,s0220_01977,Background Climate-induced coral bleaching pos...,Changes in Bleaching Susceptibility among Cora...,2013.0,PLoS ONE,180,"Biology, Medicine, Environmental Science",5,2010-2014,,,False
4,s0230_02074,Expansion of the coral genomic tool kit could ...,Can genomes predict coral bleaching?,2020.0,Science,1,"Biology, Medicine, Environmental Science, Biology",4,2020-2025,,,False


# Previewing top sentences

In [10]:
print("=== Top 10 Sentences for Annotation ===\n")
for i, row in annotation_sample.head(10).iterrows():
    print(f"[{i+1}] Score={row['climate_score']} | Year={int(row['year'])}")
    print(f"     {row['sentence']}")
    print()

=== Top 10 Sentences for Annotation ===

[1] Score=6 | Year=2014
     This estimated net C transfer to the atmosphere from permafrost thaw represents a significant factor in the overall ecosystem carbon budget of the Pan-Arctic, and a non-trivial additional contribution on top of the combined fossil fuel emissions from the eight Arctic nations over this time period.

[2] Score=5 | Year=2022
     This information on the spatial and temporal patterns on groundwater discharge at high northern latitudes is critical for predicting lake CH4 emissions in the warming Arctic, as rising temperatures, increasing precipitation, and permafrost thawing may further exacerbate groundwater CH4 inputs to lakes.

[3] Score=5 | Year=2021
     Net carbon dioxide (CO2) and methane (CH4) fluxes determine the radiative forcing contribution from these climate-sensitive ecosystems.

[4] Score=5 | Year=2013
     Background Climate-induced coral bleaching poses a major threat to coral reef ecosystems, mostly beca

# Saving output files

In [11]:
from google.colab import files

# Clean up helper columns
df_sentences_out  = df_sentences.drop(columns=['climate_score'], errors='ignore')
annotation_out    = annotation_sample.drop(columns=['year_bucket'], errors='ignore')

# Save
df_sentences_out.to_csv(CONFIG['sentences_file'], index=False)
annotation_out.to_csv(CONFIG['annotation_file'], index=False)

print(f" Saved: {CONFIG['sentences_file']} ({len(df_sentences_out)} rows)")
print(f" Saved: {CONFIG['annotation_file']} ({len(annotation_out)} rows)")

# Download both files
files.download(CONFIG['sentences_file'])
files.download(CONFIG['annotation_file'])

 Saved: climate_sentences.csv (5367 rows)
 Saved: annotation_sample_500.csv (500 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>